In [28]:
# 📌 Import necessary libraries for data handling, visualization, and modeling
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Machine Learning
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score
import sklearn.model_selection

#import classifier algorithm here
# from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier

#import preprocessing module
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import MinMaxScaler

# import evaluation metrics
from sklearn.metrics import confusion_matrix, accuracy_score
from sklearn.model_selection import cross_val_score

# Suppress warnings for better readability
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

Libraries imported successfully!


In [77]:
# 📥 Load datasets
train = pd.read_csv('Train.csv')  # Training dataset
test = pd.read_csv('Test.csv')  # Test dataset (no labels)
ss = pd.read_csv('SampleSubmission.csv')  # Sample submission format
variables = pd.read_csv('VariableDefinitions.csv')  # Data dictionary

# Display first few rows to understand structure
# print("🔹 Training Data Preview:")
# display(train.head())

# print("🔹 Variable Definitions Preview:")
# display(variables.head())

le = LabelEncoder()
train['bank_account'] = le.fit_transform(train['bank_account'])

#Separate training features from target
X_train = train.drop(['bank_account'], axis=1)
y_train = train['bank_account']
# print(y_train)

# function to preprocess our data from train models
def preprocessing_data(data):

    # Convert the following numerical labels from interger to float
    float_array = data[["household_size", "age_of_respondent", "year"]].values.astype(float)

    # categorical features to be onverted to One Hot Encoding
    categ = ["relationship_with_head",
             "marital_status",
             "education_level",
             "job_type",
             "country"]

    # One Hot Encoding conversion
    data = pd.get_dummies(data, prefix_sep="_", columns=categ)

    # Label Encoder conversion
    data["location_type"] = le.fit_transform(data["location_type"])
    data["cellphone_access"] = le.fit_transform(data["cellphone_access"])
    data["gender_of_respondent"] = le.fit_transform(data["gender_of_respondent"])

    # drop uniquid column
    data = data.drop(["uniqueid"], axis=1)

    # scale our data into range of 0 and 1
    scaler = MinMaxScaler(feature_range=(0, 1))
    data = scaler.fit_transform(data)

    return data

# preprocess the train data
processed_train = preprocessing_data(X_train)
processed_test = preprocessing_data(test)

X_train, X_test, y_train, y_test = train_test_split(processed_train, y_train, stratify = y_train, test_size = 0.1, random_state=42)

# print(y_train.value_counts())
# print(y_test.value_counts())

knn = KNeighborsClassifier(n_neighbors=40)
# knn = KNeighborsClassifier()
# cv_scores = cross_val_score(knn, X_train, y_train, cv=3)
# print each cv score (accuracy) 
# print(cv_scores)

knn.fit(X_train, y_train)

y_pred = knn.predict(X_test)
print("Error rate of LogisticRegression classifier: ", 1 - accuracy_score(y_test, y_pred))

# print((y_pred==y_test.values).sum())
# print(y_test.size)

test.bank_account = knn.predict(processed_test)

submission = pd.DataFrame({"uniqueid": test["uniqueid"] + " x " + test["country"], "bank_account": test.bank_account})
submission.sample(5)

submission.to_csv('fourth_submission.csv', index = False)

Error rate of LogisticRegression classifier:  0.11092222694432641
